# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedasker1/FlyRank_Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

*Unit of analysis: One row represents one unique content item (a specific web page) for a specific client.*

 *Time window: We are using a mid-panel month (month=2026-03) to act as our observation window, leaving the final month (June 2026) untouched as a pure, sealed holdout test set.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

*3. Features: imp_last30, clk_last30, avg_pos, max_sessions, and active_days.*

*4. Label/Proxy: is_declining (A binary target: 1 if future impressions drop by >20% compared to the feature window, 0 otherwise).*

*5. Context/Keys: client_hash_id and content_hash_id.*

*6. Excluded: future_impressions and trend_pct. Why? Because they are measured during the target window. Including them in the feature set creates a circular result (leakage) where the model learns the answer directly from the inputs*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# 1. Setup and Authenticate Hugging Face
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Base table path for the mid-panel month (March 2026)
tbl = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

print("--- 1. Query: Grain Verification ---")
# Verifying that one row is truly one content item per client per day
grain_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT client_hash_id || content_hash_id || report_date) as unique_grain
    FROM {tbl}
""").df()
print(grain_check)

print("\n--- 2. Query: Row count and date span ---")
span_check = con.sql(f"""
    SELECT
        COUNT(*) as row_count,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM {tbl}
""").df()
print(span_check)

print("\n--- 3. Query: Availability check (IS TRUE) ---")
avail_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as rows_with_ga4_true
    FROM {tbl}
""").df()
print(avail_check)

print("\n--- 4. Feature Frame (5 Features) ---")
# Splitting the month: First 15 days as features (past), rest of the month as target (future)
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as imp_last30,
        SUM(gsc_clicks) as clk_last30,
        AVG(gsc_avg_position) as avg_pos,
        MAX(ga4_sessions) as max_sessions,
        COUNT(DISTINCT report_date) as active_days
    FROM {tbl}
    WHERE report_date < '2026-03-15'
    GROUP BY 1
    HAVING imp_last30 > 100
""").df()
print(features_df.head())
print("1. imp_last30: Knowable at the decision moment because it only counts impressions before March 15.")
print("2. clk_last30: Knowable at the decision moment because clicks are strictly historical.")
print("3. avg_pos: Knowable at the decision moment because it averages past rankings.")
print("4. max_sessions: Knowable at the decision moment because it aggregates prior traffic peaks.")
print("5. active_days: Knowable at the decision moment because it tracks visibility frequency in the past window.")

print("\n--- 5. The Leakage Trap ---")
# Creating the future target window
target_df = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) as future_imp
    FROM {tbl}
    WHERE report_date >= '2026-03-15'
    GROUP BY 1
""").df()

df_model = pd.merge(features_df, target_df, on='content_hash_id').fillna(0)
# Label: Did impressions drop by more than 20%?
df_model['is_declining'] = (df_model['future_imp'] < df_model['imp_last30'] * 0.8).astype(int)

# THE TRAP: We deliberately include the future metric (future_imp) in our features
X_leaky = df_model[['imp_last30', 'clk_last30', 'avg_pos', 'max_sessions', 'active_days', 'future_imp']]
y = df_model['is_declining']

rf_leaky = RandomForestClassifier(max_depth=2, random_state=42).fit(X_leaky, y)
preds_leaky = rf_leaky.predict(X_leaky)
print(f"Leaky Score (Precision): {precision_score(y, preds_leaky):.3f} -> Looks perfect because the model cheated!")

# THE FIX: Drop the leaky column and keep the honest number
X_honest = df_model[['imp_last30', 'clk_last30', 'avg_pos', 'max_sessions', 'active_days']]
rf_honest = RandomForestClassifier(max_depth=2, random_state=42).fit(X_honest, y)
preds_honest = rf_honest.predict(X_honest)
print(f"Honest Score (Precision): {precision_score(y, preds_honest, zero_division=0):.3f} -> The real, earned baseline.")

--- 1. Query: Grain Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_grain
0     9841378       9841378

--- 2. Query: Row count and date span ---
   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31

--- 3. Query: Availability check (IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_ga4_true
0     9841378              413966

--- 4. Feature Frame (5 Features) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  imp_last30  clk_last30    avg_pos  max_sessions  \
0  content_2e6360ad20fd7107       211.0         1.0   3.941856          <NA>   
1  content_65c50dfe9d87a585      1394.0         0.0   6.104078          <NA>   
2  content_d49a012dcb924e31       237.0         0.0   4.542255          <NA>   
3  content_614baf2af4330bd7       385.0         1.0   4.351876          <NA>   
4  content_225dc9235023be5f       272.0         1.0  10.398054          <NA>   

   active_days  
0           14  
1           14  
2           14  
3           14  
4           14  
1. imp_last30: Knowable at the decision moment because it only counts impressions before March 15.
2. clk_last30: Knowable at the decision moment because clicks are strictly historical.
3. avg_pos: Knowable at the decision moment because it averages past rankings.
4. max_sessions: Knowable at the decision moment because it aggregates prior traffic peaks.
5. active_days: Knowable at the decision moment because it t

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leaky Score (Precision): 0.890 -> Looks perfect because the model cheated!
Honest Score (Precision): 0.000 -> The real, earned baseline.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

*This dataset is an unbalanced panel, meaning not all clients have tracking history that starts at the same time. Therefore, a NULL or missing row does not strictly mean "zero traffic"—it might just be before the client's tracking started. Furthermore, because this is purely observational search data, it cannot definitively prove causality (e.g., that changing a title causes a CTR increase), it can only show correlation and momentum*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.